# BirdCLEF 2026 - B2 Only Inference
Self-contained EfficientNet-B2 notebook for a quick independent submission.

In [ ]:
import time
from collections import defaultdict
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b2

DATA_DIR = Path('/kaggle/input/competitions/birdclef-2026')
B2_MODEL_PATH = Path('/kaggle/input/datasets/drakus74/efficientnet-b2-smartcrop/efficientnet_b2_SMARTCROP_BCE_20260315.pt')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

SAMPLE_RATE = 32000
WINDOW_SEC = 5.0
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512
FMIN = 50
FMAX = 14000
TEMPERATURE = 0.3
BATCH_SIZE = 32
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Device:', DEVICE)

In [ ]:
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
taxonomy_df = pd.read_csv(DATA_DIR / 'taxonomy.csv')
train_df = pd.read_csv(DATA_DIR / 'train.csv')

SPECIES = taxonomy_df['primary_label'].astype(str).tolist()
species_to_idx = {sp: i for i, sp in enumerate(SPECIES)}
TRAIN_SPECIES = sorted(train_df['primary_label'].astype(str).unique().tolist())
train_species_idx = {sp: i for i, sp in enumerate(TRAIN_SPECIES)}
ZERO_SHOT_PRIOR = 1.0 / len(SPECIES)

print(f'Taxonomy species: {len(SPECIES)}')
print(f'Train species: {len(TRAIN_SPECIES)}')

In [ ]:
model_b2 = efficientnet_b2(weights=None)
in_f = model_b2.classifier[1].in_features
model_b2.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_f, len(TRAIN_SPECIES)),
)

state_dict = torch.load(B2_MODEL_PATH, map_location=DEVICE, weights_only=True)
cleaned = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
model_b2.load_state_dict(cleaned)
model_b2 = model_b2.to(DEVICE)
model_b2.eval()

print(f'B2 loaded ✅ | features: {in_f} | train classes: {len(TRAIN_SPECIES)}')

In [ ]:
def parse_row_id(row_id):
    stem, end_time = row_id.rsplit('_', 1)
    return stem, int(end_time)

def find_audio_file(soundscape_dir, file_stem):
    for ext in ['.ogg', '.wav', '.flac']:
        candidate = soundscape_dir / f'{file_stem}{ext}'
        if candidate.exists():
            return candidate
    return None

def audio_to_mel(audio):
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT,
        hop_length=HOP_LENGTH, fmin=FMIN, fmax=FMAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_min, mel_max = mel_db.min(), mel_db.max()
    if mel_max - mel_min > 1e-8:
        mel_db = (mel_db - mel_min) / (mel_max - mel_min)
    else:
        mel_db = np.zeros_like(mel_db)
    return mel_db.astype(np.float32)

def mel_to_tensor(mel):
    tensor = torch.from_numpy(mel).unsqueeze(0).repeat(3, 1, 1)
    tensor = F.interpolate(tensor.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False).squeeze(0)
    return tensor

soundscape_dir = DATA_DIR / 'test_soundscapes'
window_samples = int(SAMPLE_RATE * WINDOW_SEC)
file_windows = defaultdict(list)

for row_id in sample_sub['row_id']:
    stem, end_time = parse_row_id(row_id)
    file_windows[stem].append((row_id, end_time))

In [ ]:
row_frames = []
t_start = time.time()

for file_idx, file_stem in enumerate(sorted(file_windows.keys()), start=1):
    audio_path = find_audio_file(soundscape_dir, file_stem)
    if audio_path is None:
        row_ids = [row_id for row_id, _ in file_windows[file_stem]]
        prior_block = np.full((len(row_ids), len(SPECIES)), ZERO_SHOT_PRIOR, dtype=np.float32)
        frame = pd.DataFrame(prior_block, columns=SPECIES)
        frame.insert(0, 'row_id', row_ids)
        row_frames.append(frame)
        continue

    audio, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
    mel_tensors = []
    row_ids = []

    for row_id, end_time in file_windows[file_stem]:
        start_sample = max(0, int((end_time - WINDOW_SEC) * SAMPLE_RATE))
        end_sample = int(end_time * SAMPLE_RATE)
        chunk = audio[start_sample:end_sample]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        elif len(chunk) > window_samples:
            chunk = chunk[:window_samples]

        mel_tensors.append(mel_to_tensor(audio_to_mel(chunk)))
        row_ids.append(row_id)

    mel_batch = torch.stack(mel_tensors)
    probs_batches = []
    for start_idx in range(0, len(mel_batch), BATCH_SIZE):
        batch = mel_batch[start_idx:start_idx + BATCH_SIZE].to(DEVICE)
        with torch.no_grad():
            logits = model_b2(batch)
            probs = torch.sigmoid(logits / TEMPERATURE).cpu().numpy()
        probs_batches.append(probs)

    probs_b2 = np.vstack(probs_batches)
    projected = np.full((probs_b2.shape[0], len(SPECIES)), ZERO_SHOT_PRIOR, dtype=np.float32)
    for species_name, train_idx in train_species_idx.items():
        taxonomy_idx = species_to_idx.get(species_name)
        if taxonomy_idx is not None:
            projected[:, taxonomy_idx] = probs_b2[:, train_idx]

    frame = pd.DataFrame(projected, columns=SPECIES)
    frame.insert(0, 'row_id', row_ids)
    row_frames.append(frame)

    if file_idx % 10 == 0:
        elapsed = time.time() - t_start
        print(f'{file_idx}/{len(file_windows)} soundscapes | {elapsed/60:.1f} min elapsed')

submission = pd.concat(row_frames, ignore_index=True)
submission = submission.set_index('row_id').reindex(sample_sub['row_id']).reset_index()
submission.columns = sample_sub.columns
submission = submission.fillna(ZERO_SHOT_PRIOR)

assert set(submission['row_id']) == set(sample_sub['row_id'])
submission.to_csv(OUTPUT_PATH, index=False)

elapsed = time.time() - t_start
print(f'Submission saved: {OUTPUT_PATH}')
print(f'Shape: {submission.shape}')
print(f'Total time: {elapsed/60:.1f} min')
submission.head()